# Premier modèle de prévision - hausse de loyer 2026

**Objectif :** tester plusieurs méthodes de prévision de la hausse 2026 avec un **même cadre de backtest** (2023, 2024, 2025),
et voir lesquelles tiennent.

**Cible (définition de la section 1 du notebook final) :** croissance annualisée du **loyer effectif** à unité constante,
médiane de tous les baux débutant dans l'année (renouvellements et relocations, six immeubles).

### Comment nos analyses se relient dans le modèle

| Notebook | Ce qu'il a établi | Rôle dans le modèle |
|---|---|---|
| `equinoxe-2026` (section 1) | la définition de la hausse | la **cible** à prévoir |
| `exploration` / section 2 | `sRentEffective` est fiable; clé d'unité; colonnes | les **données** utilisées sont valides |
| `unite_constante` | apparier, filtrer, annualiser | la **mesure** de la cible, année par année |
| `concessions` | effectif = contractuel - hausse du rabais | la **décomposition** : prévoir le contractuel, puis la hausse du rabais |
| `renouvellements_relocations` | les deux types de baux suivent des règles différentes; The Met est probablement exempté | les **segments** : renouvellements Québec, renouvellements Ontario, relocations |
| `donnees_publiques` + taux réglementaires | SCHL (inoccupation, loyers), TAL, Ontario | les **variables externes** : ancrage réglementaire et contexte de marché |

L'idée : la décomposition `effectif ≈ contractuel - hausse du rabais` permet de prévoir chaque morceau avec la variable
qui le pilote (TAL pour les renouvellements québécois, marché pour les relocations, inoccupation pour les concessions).

Un quatrième modèle (D, scikit-learn) suit **littéralement** la définition de la hausse : il prédit la croissance de
**chaque bail** de l'année à partir de son bail précédent, puis prend la **médiane**.

## 0. Configuration et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))   # évite un avertissement de joblib sous Windows
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

LEASES_PATH = "private_data/equinoxe_lease_history.csv"
CMHC_PATH = "public_data/cmhc_market_data.csv"
REGULATORY_RATES_PATH = "public_data/regulatory_rates.csv"

UNIT_KEY = ["sPropCode", "sUnitCode"]
MIN_GAP_YEARS = 0.5
MAX_GAP_YEARS = 2.5
DAYS_PER_YEAR = 365.25
PCT = 100
RENEWAL, TURNOVER = 1, 0
ONTARIO = "Ontario"
MONTREAL_MARKET = "Montréal"

FIRST_HISTORY_YEAR = 2019
BACKTEST_YEARS = [2023, 2024, 2025]
FORECAST_YEAR = 2026
MOVING_AVERAGE_YEARS = 3       # fenêtre des moyennes mobiles (modèles B et C)

SEGMENTS = ["renewal_QC", "renewal_ON", "turnover"]
OTTAWA_MARKET = "Ottawa — partie Ontario"

# Modèle D (gradient boosting au niveau des paires)
UNIT_FEATURES = ["sBeds", "sSqft", "sFloor", "gap_years", "is_renewal"]
GBM_PARAMS = {
    "loss": "absolute_error",   # erreur absolue : le modèle vise la médiane, comme notre définition
    "max_iter": 200,
    "learning_rate": 0.05,
    "max_depth": 3,             # arbres peu profonds : peu de données par année
    "min_samples_leaf": 40,     # chaque feuille repose sur au moins 40 paires
    "random_state": 0,
}
MODEL_D_PATH = Path("models/modele_d_gbm.joblib")

LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",
    "sRenewal": "is_renewal",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sTermMonths": "lease_term_months",
}

leases = pd.read_csv(LEASES_PATH, parse_dates=["sLeaseFrom", "sLeaseTo"]).rename(columns=LEASE_COLUMN_RENAMES)
leases["lease_start_year"] = leases["lease_start"].dt.year
cmhc = pd.read_csv(CMHC_PATH)
regulatory_rates = pd.read_csv(REGULATORY_RATES_PATH).set_index("year")
print(f"{len(leases):,} baux | SCHL : {cmhc['year'].min()}-{cmhc['year'].max()} | taux réglementaires : "
      f"{regulatory_rates.index.min()}-{regulatory_rates.index.max()}")

## 1. Construire l'historique annuel de la cible et de ses composantes

On reprend la méthode de `unite_constante` : chaque bail est comparé au bail précédent de la même unité, les écarts non
comparables sont filtrés, la variation est annualisée. Pour chaque paire, on calcule aussi la **hausse du rabais**
(rabais du nouveau bail - rabais du bail précédent, en points), mécanisme établi dans `concessions`.

Chaque paire est rangée dans un **segment** :
- `renewal_QC` : renouvellement dans un immeuble québécois (encadré par le TAL);
- `renewal_ON` : renouvellement à The Met (Ontario, probablement exempté du plafond);
- `turnover` : relocation (loyer de marché, sans plafond dans les deux provinces).

In [ ]:
def build_comparable_pairs(lease_df):
    """Paires de baux consécutifs d'une même unité, filtrées, avec croissances annualisées et hausse du rabais."""
    ordered = lease_df.sort_values(UNIT_KEY + ["lease_start"]).copy()
    previous = (ordered.groupby(UNIT_KEY)[["contract_rent", "effective_rent", "lease_start"]]
                .shift(1).add_prefix("prev_"))
    pairs = pd.concat([ordered, previous], axis=1).dropna(subset=["prev_lease_start"])
    pairs["gap_years"] = (pairs["lease_start"] - pairs["prev_lease_start"]).dt.days / DAYS_PER_YEAR
    is_comparable = pairs["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    pairs = pairs[is_comparable].copy()

    for rent in ["contract_rent", "effective_rent"]:
        ratio = pairs[rent] / pairs[f"prev_{rent}"]
        pairs[f"{rent}_growth_pct"] = (ratio ** (1 / pairs["gap_years"]) - 1) * PCT

    discount_now = (1 - pairs["effective_rent"] / pairs["contract_rent"]) * PCT
    discount_before = (1 - pairs["prev_effective_rent"] / pairs["prev_contract_rent"]) * PCT
    pairs["discount_change_pts"] = discount_now - discount_before
    pairs["prev_discount_pct"] = discount_before

    is_ontario = pairs["sState"] == ONTARIO
    pairs["segment"] = np.select(
        [pairs["is_renewal"].eq(RENEWAL) & ~is_ontario, pairs["is_renewal"].eq(RENEWAL) & is_ontario],
        ["renewal_QC", "renewal_ON"], default="turnover")
    return pairs


pairs = build_comparable_pairs(leases)
pairs = pairs[pairs["lease_start_year"] >= FIRST_HISTORY_YEAR]
print(f"{len(pairs):,} paires comparables depuis {FIRST_HISTORY_YEAR}")

In [ ]:
yearly_target = pairs.groupby("lease_start_year").agg(
    pairs=("effective_rent_growth_pct", "size"),
    effective_growth=("effective_rent_growth_pct", "median"),
    contract_growth=("contract_rent_growth_pct", "median"),
    discount_change=("discount_change_pts", "median"))

yearly_segments = pairs.groupby(["lease_start_year", "segment"]).agg(
    pairs=("effective_rent_growth_pct", "size"),
    contract_growth=("contract_rent_growth_pct", "median"),
    discount_change=("discount_change_pts", "median")).unstack("segment")

segment_shares = pd.crosstab(pairs["lease_start_year"], pairs["segment"], normalize="index")

print("Cible et composantes, tous baux :")
print(yearly_target.round(2), end="\n\n")
print("Par segment :")
print(yearly_segments.round(2))

**Lecture :** la cible (`effective_growth`) oscille entre 1,6 % et 4,6 % sur 2019-2025, avec un creux en 2023-2024.
L'écart entre contractuel et effectif suit la hausse du rabais : nul jusqu'en 2023, puis 1,2 point en 2024 et 3,0 points en 2025.
Les renouvellements à The Met n'existent qu'à partir de 2024 : pour les années antérieures, il n'y a pas d'historique ontarien.

## 2. Relations avec les données externes

Avant de les utiliser dans un modèle, on vérifie si les variables externes sont **réellement liées** à nos composantes.
Avec 5 à 7 années, ces relations sont **exploratoires** : elles orientent le modèle, elles ne prouvent rien.

### 2a. Renouvellements québécois et taux du TAL

Si les renouvellements suivaient le TAL, la « prime » (hausse observée - taux du TAL) serait stable d'une année à l'autre.

In [ ]:
renewal_vs_tal = pd.DataFrame({
    "renewal_QC_contract_growth": yearly_segments[("contract_growth", "renewal_QC")],
    "tal_rate": regulatory_rates["tal_unheated_pct"],
}).dropna()
renewal_vs_tal["premium_over_tal"] = renewal_vs_tal["renewal_QC_contract_growth"] - renewal_vs_tal["tal_rate"]
print(renewal_vs_tal.round(2))
print(f"\nCorrélation hausse des renouvellements QC / taux TAL : {renewal_vs_tal.iloc[:, :2].corr().iloc[0, 1]:.2f}")

**Lecture :** la relation est faible et la prime est **instable** : environ +4 points en 2021-2022 (TAL très bas,
Équinoxe augmente bien au-dessus), puis proche de 0 depuis 2023 (+0,3, -1,1, +0,5). Depuis que le TAL est élevé, les
renouvellements québécois suivent à peu près le taux du TAL; avant, ils l'ignoraient. Le taux du TAL n'est qu'une
recommandation : il ne s'impose que si le locataire conteste. Utiliser le TAL comme ancrage n'est donc défendable
que pour le **régime récent**, ce qu'on testera dans le modèle C.

### 2b. Hausse du rabais et inoccupation du marché (SCHL)

C'est l'hypothèse « vacance → concessions » testée sans succès avec les données internes (section 2g du notebook final),
reprise ici avec le taux d'inoccupation **du marché de Montréal** publié par la SCHL (enquête d'octobre).

In [ ]:
montreal = cmhc[cmhc["market"] == MONTREAL_MARKET].set_index("year")
discount_vs_vacancy = pd.DataFrame({
    "discount_change": yearly_target["discount_change"],
    "vacancy_rate": montreal["vacancy_rate"],
    "vacancy_change": montreal["vacancy_rate"].diff(),
}).dropna()
print(discount_vs_vacancy.round(2))
print(f"\nCorrélation hausse du rabais / variation de l'inoccupation (même année) : "
      f"{discount_vs_vacancy[['discount_change', 'vacancy_change']].corr().iloc[0, 1]:.2f}  (n = {len(discount_vs_vacancy)})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(discount_vs_vacancy["vacancy_change"], discount_vs_vacancy["discount_change"])
for year, row in discount_vs_vacancy.iterrows():
    ax.annotate(str(year), (row["vacancy_change"], row["discount_change"]), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("variation du taux d'inoccupation, Montréal (points, SCHL)")
ax.set_ylabel("hausse médiane du rabais (points)")
ax.set_title("Quand l'inoccupation du marché monte, les rabais augmentent")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** l'inoccupation à Montréal baisse jusqu'en 2023 (1,5 %, marché très serré), puis remonte (2,1 % en 2024,
2,9 % en 2025). Les rabais d'Équinoxe suivent : stables quand le marché se resserre, en forte hausse quand il se détend
(corrélation d'environ 0,8). C'est cohérent avec l'hypothèse de départ : **les concessions répondent au marché**.

**Prudence :** 4 points seulement, et la relation est **contemporaine** - l'inoccupation d'octobre de l'année *t* n'est
connue qu'à la fin de *t*. Elle ne peut donc pas servir directement à prévoir *t*; elle sert à construire des
**scénarios** (« si l'inoccupation continue de monter en 2026... »).

## 3. Cadre de backtest

Règle d'or : pour prévoir l'année *t*, un modèle n'a le droit d'utiliser que ce qui était **connu au début de *t*** :

- les baux débutant **avant** *t*;
- les taux réglementaires de *t* (le TAL et l'Ontario les publient avant le début de l'année);
- la SCHL jusqu'à octobre *t* - 1.

Chaque modèle est une fonction `model(history, year)` qui reçoit l'historique coupé à *t* - 1. `backtest()` coupe
l'historique, appelle le modèle et compare la prévision à la valeur réellement observée en *t*.

In [ ]:
def history_until(year):
    """Tout ce qu'un modèle a le droit de voir pour prévoir `year`."""
    known_years = yearly_target.index < year
    return {
        "target": yearly_target[known_years],
        "segments": yearly_segments[yearly_segments.index < year],
        "shares": segment_shares[segment_shares.index < year],
        "tal_rate": regulatory_rates.loc[year, "tal_unheated_pct"],
    }


def backtest(model, target_year):
    """Prévoit target_year avec l'information disponible à la fin de target_year - 1, puis compare au réel."""
    forecast = model(history_until(target_year), target_year)
    actual = yearly_target.loc[target_year, "effective_growth"]
    return {"forecast": forecast, "actual": actual, "error": forecast - actual}

## 4. Modèles candidats

| Modèle | Idée | Hypothèse |
|---|---|---|
| **A. Persistance** | la hausse de *t* = la hausse de *t* - 1 | l'année prochaine ressemble à la dernière |
| **B. Moyenne mobile** | moyenne des 3 dernières années | les chocs d'une année se compensent |
| **C. Décomposition** | par segment : contractuel prévu - hausse du rabais prévue, pondéré par la part de chaque segment | chaque morceau a son propre moteur |

Détail du modèle C, pour l'année *t* :
- **renouvellements QC (contractuel)** = taux du TAL de *t* + prime moyenne sur le TAL des 3 dernières années;
- **renouvellements ON (contractuel)** = dernière valeur observée à The Met (immeuble exempté, pas de plafond appliqué);
  à défaut d'historique ontarien, on reprend la prévision québécoise;
- **relocations (contractuel)** = moyenne des 3 dernières années;
- **hausse du rabais** = dernière valeur observée par segment (persistance);
- **pondération** = part de chaque segment l'année précédente.

Limite assumée du modèle C : combiner des médianes de segments n'est pas exactement la médiane de l'ensemble.
L'approximation est vérifiée par le backtest.

In [ ]:
def model_persistence(history, year):
    return history["target"]["effective_growth"].iloc[-1]


def model_moving_average(history, year):
    return history["target"]["effective_growth"].iloc[-MOVING_AVERAGE_YEARS:].mean()


def last_known(series):
    known = series.dropna()
    return known.iloc[-1] if len(known) else np.nan


def model_decomposition(history, year, return_details=False):
    segments = history["segments"]
    tal_premium = (segments[("contract_growth", "renewal_QC")]
                   - regulatory_rates["tal_unheated_pct"]).dropna().iloc[-MOVING_AVERAGE_YEARS:].mean()

    contract_forecast = {
        "renewal_QC": history["tal_rate"] + tal_premium,
        "turnover": segments[("contract_growth", "turnover")].dropna().iloc[-MOVING_AVERAGE_YEARS:].mean(),
    }
    ontario_last = last_known(segments[("contract_growth", "renewal_ON")])
    contract_forecast["renewal_ON"] = contract_forecast["renewal_QC"] if np.isnan(ontario_last) else ontario_last

    discount_forecast = {segment: last_known(segments[("discount_change", segment)]) for segment in SEGMENTS}
    discount_forecast = {segment: 0.0 if np.isnan(value) else value for segment, value in discount_forecast.items()}

    weights = history["shares"].iloc[-1].reindex(SEGMENTS).fillna(0)
    details = pd.DataFrame({"contract_growth": contract_forecast, "discount_change": discount_forecast,
                            "weight": weights}).loc[SEGMENTS]
    details["effective_growth"] = details["contract_growth"] - details["discount_change"]
    forecast = (details["effective_growth"] * details["weight"]).sum()
    return (forecast, details) if return_details else forecast


MODELS = {
    "A. Persistance": model_persistence,
    "B. Moyenne mobile 3 ans": model_moving_average,
    "C. Décomposition": model_decomposition,
}

### Modèle D - gradient boosting au niveau des paires (scikit-learn)

**Pourquoi ce modèle :** notre définition est la médiane des hausses **de chaque bail** de l'année. Le modèle D fait exactement
cela : il apprend la hausse d'une paire à partir de ce qu'on sait **avant** le nouveau bail, prédit chaque bail de l'année,
et prend la médiane. Il apprend sur environ 3 000 paires, et non sur 7 années.

**Variables (toutes connues avant le début du nouveau bail) :**

| Variable | Pourquoi |
|---|---|
| immeuble (indicatrices), chambres, superficie, étage | le produit loué |
| `gap_years` | durée écoulée depuis le bail précédent |
| `prev_discount_pct` | rabais du bail précédent : un rabais élevé peut reculer au bail suivant, et inversement |
| `regulatory_rate` | taux du TAL (Québec) ou ligne directrice (Ontario) de l'année, publiés avant l'année |
| `is_renewal` | renouvellement ou relocation |

**Le type de bail n'est pas connu d'avance** (le locataire restera-t-il?). Chaque bail est donc prédit deux fois, comme
renouvellement et comme relocation, pondéré par la part historique de renouvellements de son immeuble. La médiane se calcule
sur cet ensemble pondéré.

**Choix du modèle :** `HistGradientBoostingRegressor` avec une perte en erreur absolue (il vise la médiane, comme notre
définition), des arbres peu profonds et des feuilles d'au moins 40 paires pour limiter le surapprentissage.
L'inoccupation SCHL est testée en section 6b, mais pas retenue (voir pourquoi en 6b).

In [ ]:
BUILDINGS = sorted(leases["sBuilding"].unique())
SCHL_VACANCY = cmhc.set_index(["market", "year"])["vacancy_rate"]


def regulatory_rate_for(df):
    """Taux réglementaire de l'année du nouveau bail : TAL au Québec, ligne directrice en Ontario."""
    years = df["lease_start_year"]
    tal = years.map(regulatory_rates["tal_unheated_pct"])
    ontario = years.map(regulatory_rates["ontario_guideline_pct"])
    return np.where(df["sState"] == ONTARIO, ontario, tal)


def previous_october_vacancy(df):
    """Taux d'inoccupation SCHL d'octobre de l'année précédente, pour le marché de l'immeuble."""
    markets = np.where(df["sState"] == ONTARIO, OTTAWA_MARKET, MONTREAL_MARKET)
    return [SCHL_VACANCY.get((market, year - 1), np.nan) for market, year in zip(markets, df["lease_start_year"])]


def feature_matrix(df, use_prev_discount=True, use_regulatory=True, use_vacancy=False):
    features = df[UNIT_FEATURES].astype(float).copy()
    if use_prev_discount:
        features["prev_discount_pct"] = df["prev_discount_pct"]
    if use_regulatory:
        features["regulatory_rate"] = regulatory_rate_for(df)
    if use_vacancy:
        features["previous_october_vacancy"] = previous_october_vacancy(df)
    for building in BUILDINGS:
        features[f"building_{building}"] = (df["sBuilding"] == building).astype(float)
    return features


def weighted_median(values, weights):
    order = np.argsort(values)
    sorted_values, cumulative_weights = values[order], np.cumsum(weights[order])
    return sorted_values[np.searchsorted(cumulative_weights, cumulative_weights[-1] / 2)]


def predict_both_lease_types(model, population, training_pairs, feature_options):
    """Prédit chaque bail comme renouvellement et comme relocation, pondéré par la part de renouvellements de l'immeuble."""
    renewal_share = (population["sBuilding"].map(training_pairs.groupby("sBuilding")["is_renewal"].mean())
                     .fillna(training_pairs["is_renewal"].mean()).to_numpy())
    scenarios = []
    for status, weight in [(RENEWAL, renewal_share), (TURNOVER, 1 - renewal_share)]:
        scenario = population.assign(is_renewal=status)
        scenario["predicted_growth"] = model.predict(feature_matrix(scenario, **feature_options))
        scenario["weight"] = weight
        scenarios.append(scenario)
    return pd.concat(scenarios)


def fit_pair_model(training_pairs, feature_options):
    model = HistGradientBoostingRegressor(**GBM_PARAMS)
    model.fit(feature_matrix(training_pairs, **feature_options), training_pairs["effective_rent_growth_pct"])
    return model

### Population de baux à prévoir

- **Pour une année de backtest** : les baux qui ont réellement débuté cette année-là. On n'utilise que les informations de leur
  **bail précédent** (rabais, durée écoulée) et du logement, connues avant le début du nouveau bail.
- **Pour 2026** : le dernier bail connu de chaque logement se termine à une date connue (`lease_end`). Le prochain bail débute
  le lendemain. On retient les logements dont ce prochain bail débute en 2026, avec les mêmes filtres d'écart que pour l'historique.

In [ ]:
last_known_leases = leases.sort_values("lease_start").groupby(UNIT_KEY).tail(1).copy()
next_lease_start = last_known_leases["lease_end"] + pd.Timedelta(days=1)
population_2026 = last_known_leases.assign(
    lease_start_year=next_lease_start.dt.year,
    gap_years=(next_lease_start - last_known_leases["lease_start"]).dt.days / DAYS_PER_YEAR,
    prev_discount_pct=(1 - last_known_leases["effective_rent"] / last_known_leases["contract_rent"]) * PCT)
population_2026 = population_2026[
    (population_2026["lease_start_year"] == FORECAST_YEAR)
    & population_2026["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")]

print(f"{len(population_2026):,} logements dont le prochain bail débute en {FORECAST_YEAR}")
print(population_2026["sBuilding"].value_counts().to_string())
print(f"\nRabais moyen du bail précédent : population 2026 = {population_2026['prev_discount_pct'].mean():.1f} %, "
      f"baux de {FORECAST_YEAR - 1} = {pairs.loc[pairs['lease_start_year'] == FORECAST_YEAR - 1, 'prev_discount_pct'].mean():.1f} %")


def population_for(year):
    return population_2026 if year == FORECAST_YEAR else pairs[pairs["lease_start_year"] == year]


RETAINED_FEATURES = {"use_prev_discount": True, "use_regulatory": True, "use_vacancy": False}


def model_pair_gbm(history, year, feature_options=RETAINED_FEATURES):
    training_pairs = pairs[pairs["lease_start_year"] < year]
    model = fit_pair_model(training_pairs, feature_options)
    predictions = predict_both_lease_types(model, population_for(year), training_pairs, feature_options)
    return weighted_median(predictions["predicted_growth"].to_numpy(), predictions["weight"].to_numpy())


MODELS["D. Gradient boosting (paires)"] = model_pair_gbm

## 5. Résultats du backtest

Erreur = prévision - réel, en points de pourcentage. On résume par l'**erreur absolue moyenne** (MAE) sur 2023-2025 :
« en moyenne, le modèle se trompe de X points ».

In [ ]:
backtest_rows = []
for model_name, model in MODELS.items():
    for year in BACKTEST_YEARS:
        result = backtest(model, year)
        backtest_rows.append({"model": model_name, "year": year, **result})
backtest_results = pd.DataFrame(backtest_rows)

forecast_table = backtest_results.pivot(index="model", columns="year", values="forecast")
forecast_table["MAE"] = backtest_results.groupby("model")["error"].apply(lambda errors: errors.abs().mean())
forecast_table.loc["Réel observé"] = yearly_target.loc[BACKTEST_YEARS, "effective_growth"]
forecast_table.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
history_years = yearly_target.index
ax.plot(history_years, yearly_target["effective_growth"], "k-o", lw=2.5, label="réel (cible)")
for model_name in MODELS:
    model_rows = backtest_results[backtest_results["model"] == model_name]
    ax.plot(model_rows["year"], model_rows["forecast"], "--s", label=model_name)
ax.axvspan(min(BACKTEST_YEARS) - 0.5, max(BACKTEST_YEARS) + 0.5, color="grey", alpha=0.1, label="période de backtest")
ax.set_xlabel("année")
ax.set_ylabel("croissance du loyer effectif, même unité (%)")
ax.set_title("Backtest : prévisions faites avec l'information disponible l'année précédente")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :**

- **Aucun modèle ne prévoit bien 2023** : le ralentissement à environ 2 % arrive après deux années fortes (3,5 % et 4,6 %).
  Rien dans l'historique interne ne l'annonçait.
- **Les modèles simples font mieux que la décomposition** (MAE de 1,36 et 1,54 point contre 2,18).
  La décomposition surestime systématiquement : la prime sur le TAL qu'elle utilise est gonflée par 2021-2022,
  quand Équinoxe augmentait bien au-dessus d'un TAL très bas (section 2a). Ce régime n'existe plus.
- Avec 3 années de backtest seulement, les écarts entre modèles sont indicatifs, pas définitifs.

## 6. Prévisions 2026

On applique chaque modèle avec tout l'historique jusqu'à 2025, et les taux réglementaires 2026 (TAL : 3,1 %; Ontario : 2,1 %,
non appliqué à The Met si l'exemption se confirme).

In [ ]:
def estimate_2026(model):
    """Hausse 2026 du loyer effectif à unité constante, en %, selon le modèle donné."""
    return model(history_until(FORECAST_YEAR), FORECAST_YEAR)


forecasts_2026 = pd.Series({name: estimate_2026(model) for name, model in MODELS.items()}, name="prévision 2026 (%)")
forecasts_2026.to_frame().assign(MAE_backtest=forecast_table["MAE"]).round(2)

In [ ]:
_, decomposition_details_2026 = model_decomposition(history_until(FORECAST_YEAR), FORECAST_YEAR, return_details=True)
print("Détail du modèle C pour 2026 :")
decomposition_details_2026.round(2)

### Scénarios sur les concessions

La plus grande incertitude pour 2026 est l'évolution des rabais (section 2b). On fait varier **uniquement** la hausse du rabais
dans la décomposition, en gardant les hausses contractuelles du modèle C :

- **bas** : les rabais continuent d'augmenter au rythme de 2025;
- **central** : les rabais augmentent moitié moins qu'en 2025;
- **haut** : les rabais se stabilisent (hausse nulle).

In [ ]:
DISCOUNT_SCENARIOS = {"bas (rabais comme 2025)": 1.0, "central (moitié de 2025)": 0.5, "haut (rabais stables)": 0.0}

scenario_rows = {}
for scenario_name, share_of_last_change in DISCOUNT_SCENARIOS.items():
    scenario = decomposition_details_2026.copy()
    scenario["effective_growth"] = scenario["contract_growth"] - share_of_last_change * scenario["discount_change"]
    scenario_rows[scenario_name] = (scenario["effective_growth"] * scenario["weight"]).sum()
pd.Series(scenario_rows, name="hausse effective 2026 (%)").round(2).to_frame()

### 6b. Modèle D : quelles variables comptent?

On refait le backtest et la prévision 2026 du modèle D en retirant ou en ajoutant **un groupe de variables à la fois**.
Ça montre ce que chaque groupe apporte, et ce qui pousse la prévision 2026.

In [ ]:
FEATURE_VARIANTS = {
    "D retenu (logement + rabais précédent + taux réglementaire)": RETAINED_FEATURES,
    "+ inoccupation SCHL": {"use_prev_discount": True, "use_regulatory": True, "use_vacancy": True},
    "sans rabais précédent": {"use_prev_discount": False, "use_regulatory": True, "use_vacancy": False},
    "sans taux réglementaire": {"use_prev_discount": True, "use_regulatory": False, "use_vacancy": False},
    "logement seulement": {"use_prev_discount": False, "use_regulatory": False, "use_vacancy": False},
}

variant_rows = {}
for variant_name, options in FEATURE_VARIANTS.items():
    variant_model = lambda history, year, options=options: model_pair_gbm(history, year, options)
    row = {year: backtest(variant_model, year)["forecast"] for year in BACKTEST_YEARS}
    row["MAE"] = np.mean([abs(row[year] - yearly_target.loc[year, "effective_growth"]) for year in BACKTEST_YEARS])
    row[FORECAST_YEAR] = estimate_2026(variant_model)
    variant_rows[variant_name] = row
pd.DataFrame(variant_rows).T.round(2)

**Lecture :**

- **Le rabais du bail précédent est la variable la plus utile au backtest** : sans elle, l'erreur passe de 1,18 à 1,45 point.
  Le modèle a appris qu'après un rabais élevé, le loyer effectif du bail suivant monte davantage (le rabais tend à reculer),
  et qu'après un bail sans rabais, il monte peu (un rabais apparaît).
- **Mais c'est aussi elle qui pousse la prévision 2026 vers le haut.** Les logements qui renouvellent en 2026 partent d'un rabais
  record (9,0 % en moyenne, contre 5,8 % pour les baux de 2025) : le modèle **extrapole** au-delà de ce qu'il a vu et suppose
  implicitement que les rabais vont reculer. Sans cette variable, il prévoit 2,3 % au lieu de 3,8 %.
- **L'inoccupation SCHL n'améliore presque pas le backtest** (1,13 contre 1,18 point, seule 2025 change) mais fait bondir
  la prévision 2026 à 5,1 % : l'inoccupation d'octobre 2025 (2,9 % à Montréal) dépasse tout ce que le modèle a vu, et il
  n'a que 4 années pour en apprendre l'effet. Gain négligeable, extrapolation forte : on ne la retient pas.
- **Le taux réglementaire aide un peu** (1,18 contre 1,29 sans lui).

**En clair :** la prévision 2026 du modèle D (3,8 %) repose sur l'hypothèse que les rabais records de 2025 vont partiellement
reculer. C'est plausible (le rabais d'un bail a tendance à revenir vers la moyenne), mais ce n'est pas garanti :
c'est exactement l'incertitude des scénarios de la section 6.

### 6c. Prévision 2026 par immeuble et par nombre de chambres (modèle D)

Bonus des consignes. Le modèle final est entraîné sur toutes les paires jusqu'en 2025, puis sauvegardé
(livrable « modèle entraîné »). Les effectifs indiquent combien de baux de 2026 entrent dans chaque médiane.

In [ ]:
final_training_pairs = pairs[pairs["lease_start_year"] < FORECAST_YEAR]
final_model_d = fit_pair_model(final_training_pairs, RETAINED_FEATURES)
predictions_2026 = predict_both_lease_types(final_model_d, population_2026, final_training_pairs, RETAINED_FEATURES)


def weighted_median_by(df, group_column):
    rows = {}
    for group_value, group in df.groupby(group_column):
        rows[group_value] = {
            "baux_2026": len(group) // 2,   # chaque bail apparaît deux fois (renouvellement et relocation)
            "hausse_effective_prevue_pct": weighted_median(group["predicted_growth"].to_numpy(),
                                                           group["weight"].to_numpy()),
        }
    return pd.DataFrame(rows).T


print("Par immeuble :")
print(weighted_median_by(predictions_2026, "sBuilding").round(2), end="\n\n")
print("Par nombre de chambres :")
print(weighted_median_by(predictions_2026, "sBeds").round(2))

MODEL_D_PATH.parent.mkdir(exist_ok=True)
joblib.dump({"model": final_model_d, "features": RETAINED_FEATURES, "buildings": BUILDINGS}, MODEL_D_PATH)
print(f"\nModèle sauvegardé : {MODEL_D_PATH}")

## 7. Ce que ce premier modèle nous apprend

**Prévisions 2026 (loyer effectif, même unité) :**

| Approche | 2026 | Erreur moyenne au backtest |
|---|---|---|
| A. Persistance | 3,6 % | 1,54 point |
| B. Moyenne mobile 3 ans | 2,5 % | 1,36 point |
| C. Décomposition | 1,2 % | 2,18 points |
| **D. Gradient boosting (paires)** | **3,8 %** | **1,18 point** (meilleur) |
| Scénarios sur les rabais (décomposition) | 1,2 % / 2,7 % / 4,2 % | bas / central / haut |

**Ce qu'on retient :**

1. **Le modèle D est le meilleur au backtest** et c'est celui qui suit littéralement notre définition (médiane des hausses
   de chaque bail). Il prend aussi en compte la composition réelle des baux de 2026 (937 logements connus d'avance)
   et donne une prévision par immeuble (3,4 % à The Met, 3,6 à 3,9 % ailleurs) et par nombre de chambres.
2. **Aucun modèle ne prévoit bien 2023** (tous autour de 3,6 à 4,6 % pour 2,0 % réel) : l'historique n'annonçait pas ce ralentissement.
3. **Toute l'incertitude de 2026 porte sur les rabais.** Le loyer contractuel est assez prévisible (environ 3 % pour les
   renouvellements québécois avec le TAL 2026 de 3,1 %, environ 5 % pour les relocations). La prévision effective va de 1,2 %
   si les rabais continuent de grimper comme en 2025, à environ 4 % s'ils se stabilisent ou reculent. Le modèle D (3,8 %)
   suppose un recul partiel des rabais records de 2025.
4. **L'inoccupation SCHL explique bien la hausse passée des rabais** (section 2b), mais avec 4 années elle ne peut pas servir de
   variable de modèle : on l'utilise pour choisir entre les scénarios.
5. **Prudence :** 3 années de backtest seulement; les écarts entre modèles sont indicatifs.

**Pistes pour la version 2 :**
- combiner B et D (moyenne des deux) et vérifier si la combinaison réduit l'erreur;
- construire un intervalle de prévision à partir des erreurs du backtest;
- documenter l'hypothèse sur les rabais 2026 avec l'inoccupation SCHL d'octobre 2025 et le marché;
- confirmer l'exemption de The Met (première occupation après le 15 novembre 2018).